In [8]:
import sglang as sgl
import tiktoken

In [9]:
sgl.set_default_backend(sgl.RuntimeEndpoint("http://127.0.0.1:30000"))

In [10]:
@sgl.function
def analyze_review(s, review_text):
    s += sgl.system(
        '你是一个电商评论分析助手。请严格按以下JSON格式提取信息，不要任何额外说明：{"rating": 1-5的整数, "sentiment": "positive/neutral/negative", "suggestion": "不超过15字的改进点"}'
    )
    s += sgl.user(f"评论内容：{review_text}")
    s += sgl.assistant(
        sgl.gen(
            "result",
            max_tokens=100,
            # 正则确保：{"rating":3,"sentiment":"positive","suggestion":"包装太差"}
            regex=r'\{\s*"rating"\s*:\s*[1-5]\s*,\s*"sentiment"\s*:\s*"(positive|neutral|negative)"\s*,\s*"suggestion"\s*:\s*".{1,15}"\s*\}',
        )
    )

In [11]:
state = analyze_review.run(
    review_text="耳机音质不错，但充电盒太容易划伤，希望改进包装。",
    temperature=0.1,  # 降低随机性，提升结构稳定性
)

print(state["result"])

{"rating": 3, "sentiment": "neutral", "suggestion": "改进包装"}
